# LHS 712 Final Project

**Term**: Winter, 2022

**Author**:  Carolina Chung

**Description**: information extraction tool for [clinical trial]() data. 

**Dataset**: collection of `.xml` files downloaded from the Clinical Trials database. The following filters were applied to attain this set: 
* Recruitment = completed
* Study Type = interventional
* Key Words = "Gram-Negative Bacterial Infections"

**Task**: scan each file and extract information for the following fields: 
* 
* 
* 
* 
* 
* 

**Deliverable**: summary of the extracted information as a tabular output. 

## 0. Initialize environment

In [4]:
# Load packages
import os
import re
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report

# User-defined functions
def xtract_xml(f):
    '''
    xtract_xml():   for extracting information from XML fields
    INPUT: 
            f:      file path
    OUTPUT: 
            out:    dictionary containing information for the following keys
                - ID:               study NCT ID
                - Disease:          disease of interest
                - Intervention:     interventions (e.g. treatments) used in the study
                - N:                number of interventions investigated in the study
                - Phase:            clinical trial phase
                - PMID:             PMID for any publications affiliated with the study
                - Summary:          free text providing a brief summary of the study
                - Description:      free text providing a detailed description of the study
    '''
    # Import XML data
    tree = ET.parse(f)
    root = tree.getroot()
    # Extract information
    ID = root.find('id_info').find('nct_id').text
    PMID, disease, treatment = [], [], []
    for reference in root.findall('reference'): 
        try: 
            PMID.append(reference.find('PMID').text)
        except AttributeError: 
            pass
    for condition in root.findall('condition'): 
        disease.append(condition.text)
    for intervention in root.findall('intervention'): 
        treatment.append(intervention.find('intervention_name').text)
    phase = root.find('phase').text
    summary = root.find('brief_summary').find('textblock').text
    try: 
        description = root.find('detailed_description').find('textblock').text
        description = ' '.join(re.findall(r'[a-zA-Z0-9.]+', description))
    except AttributeError: 
        description = ''
    # Define output
    out = {
        'ID': ID, 
        'Condition': disease, 
        'Intervention': treatment, 
        'N': len(treatment), 
        'Phase': phase, 
        'PMID': PMID, 
        'Summary': ' '.join(re.findall(r'[a-zA-Z0-9.]+', summary)), 
        'Description': description
    }
    return out

def read_file(f):
    '''
    read_file():        reads a text file containing ID, TXT, and LABEL information
    INPUT: 
            - f:        text file to read
    OUTPUT: 
            - out:      containing information for the following keys
                - ID:       list of IDs in the given file
                - text:      list of textual data in the given file
                - labels:   list of labels in the given file
    '''
    assert ['ID\tSummary\tLabel\n'] == open(f, 'r').readlines(1)
    data = open(f, 'r').readlines()[1:]
    id = [i.split('\t')[0].strip() for i in data]
    txt = [i.split('\t')[1].strip().split(' ') for i in data]
    labels = [i.split('\t')[2].strip().split(' ') for i in data]
    return {'ID': id, 'text': txt, 'labels': labels}

def word2feat(word, bias:float=1.0, n:int=10):
    '''
    word2feat():        reads a text file containing ID, TXT, and LABEL information
    INPUT: 
            - word:     text file to read
            - bias:     value to use for the bias feature
            - n:        number of neighboring words to inspect
    OUTPUT: 
            - features: dictionary of feature information; specifically: 
                - bias:             bias weight
                - word.lower():     lowercase of words
                - word[-n:]:        n neighboring words
                - word.isupper():   whether the word is uppercase
                - word.istitle():   whether the word is in title form
                - word.isdigit():   whether the word is a digit

    '''
    features = {
        'bias': bias,
        'word.lower()': word.lower(),
        'word[-n:]': word[-n:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit()
    }
    return features
    
def text2feat(text, bias:float=1.0, n:int=10):
    '''
    text2feat():    converts textual data into features for ML use
    INPUT:          a list of text data
    OUTPUT:         a list of feature data 
    * see description for word2feat() function for parameter details
    '''
    return [word2feat(i, bias=bias, n=n) for i in text]

def search_txt(txt, pattern, exclude=None, n:int=0):
    '''
    search_txt():       searches for text and retrieves n words in either side of the text
    INPUT: 
            - txt:      text to search
            - pattern:  substring to search for in text
            - exclude:  if this substring is found, skip given text
            - n:        number of neighboring words to extract along with match (default: n = 0)
    OUTPUT: 
            - match:    substring of text containing match along with n neighboring words
    '''
    if exclude is not None and isinstance(exclude, list) is False: 
        exclude = [exclude]
    if exclude is not None and any(x in txt for x in exclude): 
        return ''
    else: 
        word = r'\W*([\w]+)'
        match = re.search(r'{}\W*{}{}'.format(word*n, pattern, word*n), txt)
        try: 
            s = match.group().split()[-2:]
            if n == 0:
                return ' '.join(s)
            else: 
                return match
        except AttributeError: 
            return ''

def scan_txt(txt, keywords, exclude=None, n:int=0):
    '''
    scan_txt():         scans given text and returns True if any keyword is present
    INPUT: 
            - txt:      text to scan
            - keywords: list of keywords to scan for in text
            - exclude:  if this substring is found, skip given text
    OUTPUT: 
            - out:      Boolean indicating whether a keyword is in the given text
    '''
    if exclude is not None and isinstance(exclude, list) is False: 
        exclude = [exclude]
    if exclude is not None and any(x in txt for x in exclude): 
        return False
    else: 
        if isinstance(keywords, list) is False: 
            keywords = [keywords]
        out = any(key in txt for key in keywords)
        return out

## 1. Initial file scanning

In [5]:
# Define file directory
DIR = './CT_GramNeg_Complete_Interventional/'
SAVE_DATA = False

# Initial extraction
CT_list = []
for file in tqdm(os.listdir(DIR), desc='Reading XML files'): 
    if file.endswith('.xml'): 
        CT_list.append(xtract_xml(DIR + file))

Reading XML files: 100%|██████████| 744/744 [03:42<00:00,  3.34it/s]


## 2. Scan free text using regular expression
Extract pathogen information (if any) using regular expression

In [6]:
# Define regular expression patterns
pattern = '([A-Z]\. [a-z]{4,})'
# pattern = '\w+ ([A-Z][a-z]{2,} [a-z]{2,})|([A-Z]\. [a-z]{2,})'
exclude = ['vaccine', 'vaccination', 'conjugate']

# Convert extracted data into dataframe
df = pd.DataFrame(CT_list)
df = df.astype({'ID': 'string', 'Phase': 'string', 'Summary': 'string', 'Description': 'string'})

# Scan summary for pathogen
df['Pathogen'] = [search_txt(s, pattern, exclude=exclude) for s in df['Summary']]

# Scan summary for resistance info
keywords = ['resistant', 'resistance']
df['Resistance'] = [scan_txt(s, keywords, exclude=exclude) for s in df['Summary']]

# Scan summary for combination status
keywords = ['combination', 'combined']
df['Combination'] = [scan_txt(s, keywords, exclude=exclude) for s in df['Summary']]

# Scan summary for sequential status
keywords = ['sequential', 'followed by']
df['Sequential'] = [scan_txt(s, keywords, exclude=exclude) for s in df['Summary']]

# Set Combination = True if Sequential = True
df['Combination'] = df[['Combination', 'Sequential']].any(axis='columns')

## 3. Split and save free text for ML application (DEPRECATED)
Extract summary free text and split into train + test sets. Of note, make sure to include balanced levels of entries that contain information on the keywords `dose` and `combination`. 

In [9]:
# Add base labels (O)
df['Label'] = [' '.join('O'*(s.count(' ')+1)) for s in df['Summary']]

# Scan summary free text for keywords
keywords = ['dose', 'combination']
df1 = df[df['Summary'].str.contains('|'.join(keywords), regex=True)]
df0 = df[~df['Summary'].str.contains('|'.join(keywords), regex=True)]

# Split into train/test sets while balancing entries containing keywords
df1_train, df1_test = train_test_split(df1[['ID', 'Summary', 'Label']], test_size=0.2)
df0_train, df0_test = train_test_split(df0[['ID', 'Summary', 'Label']], test_size=0.2)
df_train, df_test = pd.concat([df1_train, df0_train]), pd.concat([df1_test, df0_test])
df_train, df_test = df_train.sample(frac=1), df_test.sample(frac=1)

# Save train/test splits into text data
if SAVE_DATA is True: 
    df_train.to_csv('./CT_summary_train.txt', sep='\t', index=False)
    df_test.to_csv('./CT_summary_test.txt', sep='\t', index=False)

*** MANUALLY INSPECT `Label` COLUMN AND ASSIGN LABELS FOR `Combination (C)` CLASS ***

## 4. Extract information using ML (DEPRECATED)
Attempt to extract information on `Combination` using Conditional Random Fields (CRF) model. 

In [11]:
# Load data
train_data = read_file('./Archived/CT_summary_train.txt')
test_data = read_file('./Archived/CT_summary_test.txt')

# Prepare data
X, y = [text2feat(txt) for txt in train_data['text']], train_data['labels']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
X_test, y_test = [text2feat(txt) for txt in test_data['text']], test_data['labels']

# Train model
model = CRF()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
report_val = flat_classification_report(y_val, y_pred)
print('Printing CRF model training results:')
print(report_val)

# Evaluate model
y_pred = model.predict(X_test)
report_test = flat_classification_report(y_test, y_pred)
print('Printing CRF model evaluation results:')
print(report_test)

Printing CRF model training results:
              precision    recall  f1-score   support

           C       0.00      0.00      0.00         3
           O       1.00      1.00      1.00     11715

    accuracy                           1.00     11718
   macro avg       0.50      0.50      0.50     11718
weighted avg       1.00      1.00      1.00     11718

Printing CRF model evaluation results:
              precision    recall  f1-score   support

           C       0.00      0.00      0.00        30
           O       1.00      1.00      1.00     14790

    accuracy                           1.00     14820
   macro avg       0.50      0.50      0.50     14820
weighted avg       1.00      1.00      1.00     14820



c:\Users\carol\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\carol\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\carol\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

## 5. Extract combination status using regular expression (DEPRECATED)

In [12]:
# Define regular expression patterns
# pattern = '(combined|combination)'
pattern = '[^.]* (combined|combination) [^.]*\.'
exclude = ['vaccine', 'vaccination']

# Scan summary for combination status
df['Combination'] = [search_txt(s, pattern, exclude) for s in df['Summary']]

In [13]:
df.loc[df['Combination'] != '', df.columns != 'Label']

,ID,Condition,Intervention,N,Phase,PMID,Summary,Description,Pathogen,Resistance,Combination,Sequential
13,NCT00014950,"[Cystic Fibrosis, Lung Disease, Pseudomonas In...",[CF newborn screening],1,N/A,"[85057, 6634283, 2673641, 9395429, 10959441, 1...",Although cystic fibrosis CF is the most common...,,,False,the country.,True
38,NCT00140296,"[Pregnancy, Chlamydia]",[Contraceptive counseling],1,N/A,[14969669],Consistent and correct use of an effective con...,Consistent and correct use of an effective con...,,False,and consistency.,False
73,NCT00258908,"[Diphtheria, Tetanus, Pertussis]","[Diphteria, tetanus, and Acellular Pertussis v...",1,Phase 3,[],To assess the immunogenicity profile of ADACEL...,,,False,after administration.,False
210,NCT00635128,"[Poliomyelitis, Diphtheria, Tetanus, Acellular...",[Boostrix-Polio],1,Phase 4,[22485049],Subjects aged 9 to 13 years who participated i...,,,False,and reactogenicity.,False
238,NCT00723502,"[Gram-Negative Bacterial Infections, Helicobac...","[Finafloxacin + Amoxicillin, Finafloxacin + Es...",2,Phase 2,[],The primary objective of this study is to comp...,,H. pylori,False,or Esomeprazole.,False
276,NCT00926796,[Gonorrhoea],"[Azithromycin, Gentamicin, Gemifloxacin]",3,Phase 4,[],The purpose of this study is to learn how to b...,Infection with Neisseria N. gonorrhoeae carrie...,,False,2 antibiotics.,False
293,NCT01032655,[Helicobacter Pylori Infection],[susceptibility test guided sequential therapy],1,Phase 4,[],Background Helicobacter pylori infection has b...,,H. pylori,True,triple therapy.,True
406,NCT01530672,"[HIV, Syphilis]",[MBIO POC combined HIV syphilis test ( SnapEsi)],1,N/A,[],This is a diagnostic validation study for a co...,,,False,USA .,False
540,NCT02303587,[Mycoplasma Pneumoniae Pneumonia],[methylprednisolone],1,N/A,"[15646048, 18420275, 16119439, 22778084, 21395...",The study is designed to investigate differenc...,Mycoplasma pneumonia pneumonia MPP accounts fo...,,False,with azithromycin.,True
568,NCT02454816,"[Syphilis, HIV]","[Single HIV RDT and single syphilis RDT, Dual ...",2,N/A,[],Global and regional initiatives have been laun...,Methods Desing Cluster randomized trial includ...,,False,stick blood.,False


## 6. Format final output (tabular data)

In [14]:
# Full (unnested + complete) dataframe
df_full = df.explode('Condition')
df_full = df_full.explode('Intervention')
df_full = df_full.explode('PMID')

# Save output (if prompted)
SAVE_OUTPUT = False
if SAVE_OUTPUT is True: 
    df_full.to_csv('./CT_full_output.csv', index=False)

## 7. Inspect scanned parts

In [15]:
# Instantiate filtered dataframe objects
dfP = df.loc[df['Pathogen'] != '', ['Pathogen', 'Summary']]
dfR = df.loc[df['Resistance'] == True, ['Pathogen', 'Summary']]
dfC = df.loc[df['Combination'] == True, ['Intervention', 'Summary']]
dfS = df.loc[df['Sequential'] == True, ['Intervention', 'Summary']]

In [16]:
# Print final output
df

,ID,Condition,Intervention,N,Phase,PMID,Summary,Description,Pathogen,Resistance,Combination,Sequential,Label
0,NCT00000120,"[Chlamydia Infections, Ophthalmia Neonatorum]","[Erythromycin Ointment, Silver Nitrate Drops]",2,Phase 3,"[8356971, 8233733]",To compare the effectiveness of silver nitrate...,Sexually transmitted diseases are a major caus...,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
1,NCT00000432,"[Lyme Disease, Tick-Borne Diseases]",[Education about disease prevention],1,Phase 3,"[9236962, 11275450]",This is a large study of an educational progra...,This is a large randomized trial of a Lyme dis...,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
2,NCT00000648,"[HIV Infections, Neurosyphilis]","[Penicillin G potassium, Ceftriaxone sodium]",2,N/A,[],To provide information on the response of HIV ...,Studies suggest that syphilis treatment failur...,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
3,NCT00000937,[Lyme Disease],[Antibiotics],1,Phase 3,[],The purpose of this study is to see how well a...,You will be assigned randomly like tossing a c...,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
4,NCT00000938,[Lyme Disease],"[ceftriaxone, doxycycline]",2,Phase 3,[],Lyme disease is the most common tick borne dis...,Sixty six patients will be enrolled in this st...,B. burgdorferi,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,NCT04830371,[Typhoid Fever],"[EuTCV (Vi-CRM Typhoid conjugate vaccine), Typ...",2,Phase 2/Phase 3,[],This is an observer blinded comparative single...,,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
740,NCT04865497,"[Dysentery, Dysentery, Bacillary]",[S.Flexneriza-S.Sonnei Bivalent Conjugate Vacc...,4,Phase 2,[],The purpose of this study is to evaluate immun...,,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
741,NCT04870138,[Gonococcal Infection],"[Azithromycin, Ceftriaxone, Ciprofloxacin, Nei...",5,Phase 1,[],This is a Phase 1 interventional non randomize...,This is a Phase 1 interventional non randomize...,,False,,False,O O O O O O O O O O O O O O O O O O O O O O O ...
742,NCT05216744,"[Neisseria Gonorrhoeae Infection, Chlamydia Tr...","[Ceftriaxone 1000mg + doxycycline 100 mg, Cefi...",2,Phase 2,[],The frequency of Chlamydia trachomatis and Nei...,Gonococcal infections including urethritis cer...,,False,or ceftriaxone.,False,O O O O O O O O O O O O O O O O O O O O O O O ...


In [ ]:
# Hand-pick correct + error cases
ix = [666, 11, 632]
for i in ix: 
    print(df.loc[i, 'Summary'])
    print(df.loc[i, ])

Helicobacter pylori H. pylori is the most common chronic bacterial infection in humans. The prevalence of H. pylori is about 30 50 in the Western adult population. It is estimated that about 50 of people are infected with this bacterium in Taiwan. Many studies have shown that H. pylori is an important causal factor of chronic gastritis peptic ulcer disease gastric cancer and gastric lymphoma. The World Health Organization classified H. pylori as a Group 1 carcinogen in 1994. Endoscopic examination is indicated to confirm the above diagnosis for patient with H. pylori infection. Eradication of H. pylori infection reduces the risk of gastric cancer and recurrence of peptic ulcer disease. However the eradication rate of clarithromycin based triple therapy has been declining in recent years probably related to the increasing resistant rate to clarithromycin. Several strategies have been proposed to overcome the declining eradication rate including 1 extending the treatment duration of trip